# 开放权重模型仓库审计：结构识别与生产证据

> **本章定位**：以本地开放权重模型仓库为审计对象，从模型卡、许可证、配置、Tokenizer、权重文件头与自定义代码中建立可复核的模型身份、结构和输入协议，并形成进入评估前的生产证据。

> **章节边界**：本章属于跨方向专题：开放权重模型审计，以 `31` 的 Decoder-only GPT 为基础；模型质量、部署拓扑与完整安全治理分别由 `50`、`60` 与 `70` 承接，本章不以静态文件审计替代这些阶段。

**本章总览**：模型家族信息以 2026-08-09 为快照日期。型号、上下文、许可证与 Kernel 支持具有时效性；可迁移的方法是固定来源并计算文件哈希，对本地仓库执行只读审计，并把已观察证据、冲突与不可验证项写入 `Model Audit Manifest`。

```mermaid
flowchart LR
    G["31：Decoder-only GPT"] --> I["固定来源并计算文件哈希"]
    I --> R["只读盘点本地仓库"]
    R --> C["静态解析 Config 与 Tokenizer"]
    C --> W["核验 Safetensors 与 Index"]
    W --> A["审查 Adapter、量化与 auto_map"]
    A --> M["Model Audit Manifest"]
    M --> E["50：模型评估"]
    E --> D["60：受控部署"]
    D --> S["70：安全验收与最终放量"]
```


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 跨方向专题：开放权重模型审计 |
| 本章定位 | 审计开放权重模型的本地仓库，识别声明结构、实际张量、输入协议、派生制品与供应链边界。 |
| 先修知识 | 完成 `31`；建议了解 `40`。 |
| 预计时间 | 3～4 小时 |
| 运行资源 | CPU 与容纳本地制品的磁盘；静态审计不下载模型、不加载权重，也不需要 GPU。 |
| 输入 | 固定来源与 Commit SHA 的本地模型目录、模型卡、许可证和使用政策。 |
| 交付物 | `Model Audit Manifest`、配置—权重一致性报告、许可证证据清单和运行时待验项。 |

### 1.1．学习目标

完成本章后，读者能够区分开放源码、开放权重与 API-only，能够在不执行仓库代码、不反序列化未知文件且不加载权重张量的条件下，从本地文件还原模型的声明结构、输入协议与实际张量组织，并能把证据、冲突和未知项固化为审计 Manifest。本章仅实现 RMSNorm、Top-k 路由与 KV Cache 成本模型；推理训练、MoE 并行和推理优化分别由 `A30_reasoning_model.ipynb`、`A40_training_optimization.ipynb` 与 `A50_inference_optimization.ipynb` 承接。


## 2．直觉与输入输出契约

本章采用三个独立标签：**开放权重**表示可取得参数文件；**开放代码**表示相应代码按许可证发布；**API-only** 表示只能调用托管服务。权重可下载不代表训练数据、数据治理和预训练流水线全部开放，也不自动代表可无条件商用。开放性结论应绑定精确的 `repo_id + LICENSE + 文件哈希`，品牌名称本身不足以形成长期有效的授权判断。

| 判断维度 | 要读取的证据 | 主要生产影响 |
|---|---|---|
| Dense / MoE | 总参数、激活参数、专家数与 Top-k | 显存、通信、吞吐和路由稳定性 |
| Attention / State Space | MHA/GQA/MLA、稀疏/线性/滑窗/混合注意力 | KV Cache、长上下文质量、Kernel 兼容性 |
| 训练与交互协议 | Base/Instruct/Reasoning、Chat Template、Tool Schema、Thinking 开关 | 输出格式、工具调用、延迟与回归测试 |
| 模态 | 文本、图像、视频、音频的输入输出集合 | Processor、媒体预算、服务路由与安全面 |
| 制品 | 权重精度、量化、Tokenizer/Processor、远程代码 | 显存、供应链、可复现性与回滚 |
| 治理 | 模型/代码许可证、AUP、地域/规模/用途限制 | 商用、再分发、衍生训练与法律责任 |

`active parameters` 主要描述每个 Token 的计算量，**不等于权重显存**；没有专家卸载或分片时，仍需容纳总权重。标称上下文长度也不等于业务可承受长度，必须实测 KV/状态缓存、Prefill 延迟和长上下文质量。

<!-- diagram:open-model-release-gate -->

![架构图：开放权重模型的计算状态、显存、服务容量与发布门禁关系](assets/figures/E10_open_model/open-model-release-gate.svg)

[TikZ 源文件](assets/figures/E10_open_model/open-model-release-gate.tex)


### 2.1．主要家族图谱（2026-08-09 快照）

下表汇总代表性观察入口，用于比较架构与制品边界，不构成能力排序。

| 地区 | 家族与代表检查点 | 架构观察维度 | 开放状态与生产边界 |
|---|---|---|---|
| 中国 | **Qwen**：Qwen3、Qwen3.6 | Dense/MoE 尺度谱系、Thinking/Non-thinking、线性与全注意力混合、原生多模态 | 当前开放权重主线为 Apache-2.0；小型号适合实操，MoE 的激活量不能替代总权重容量规划 |
| 中国 | **DeepSeek**：V3、R1、V4 | MLA、DeepSeekMoE、MTP、推理后训练与稀疏注意力演进 | 开放权重；逐代核验模型许可。旗舰是数据中心级，本章仅进行配置与架构审计，或使用小型蒸馏检查点 |
| 中国 | **Kimi**：K2、K3 | 大规模 MoE、MLA/KDA、Attention Residuals、Agent 与原生视觉 | K2 与 K3 许可证不同；K3 为自定义许可且规模巨大，不能把 K2 的 Modified MIT 套到 K3 |
| 中国 | **GLM**：GLM-4.5、GLM-5.2 | Hybrid Reasoning、MoE、稀疏注意力和推测解码 | 开放权重；权重与工程代码可能采用不同许可证，生产需分别固定版本 |
| 中国 | **MiniMax**：M2、M3 | MoE、稀疏/线性注意力、Agent 与原生多模态 | 开放权重但使用自定义社区许可；远程代码必须计算摘要、审计并隔离运行 |
| 美国 | **Llama**：Llama 4 Scout/Maverick | 原生图文、Early Fusion、MoE | Llama Community License + AUP，不是 Apache/MIT；地域、规模、分发与衍生条款必须审查 |
| 美国 | **Gemma**：Gemma 3/4 | 轻量到中型、Dense/MoE、局部/全局混合注意力与多模态 | 新旧代许可证不同；Gemma 4 为 Apache-2.0，不能倒推到旧检查点 |
| 美国 | **Phi**：Phi-4 系列 | 小语言模型、高质量数据/后训练、文本与紧凑多模态分支 | MIT 开放权重；适合端侧/低延迟，但任务、语言和知识边界需单独评测 |
| 美国 | **gpt-oss**：20b/120b | 稀疏 MoE、可配置推理强度、Harmony 响应格式与 Agent 工具协议 | Apache-2.0 开放权重；仅文本，必须按 Harmony 模板使用，工具执行仍需受控 Runtime |
| 美国 | **Nemotron**：Nemotron 3 | Mamba-2 + Attention + LatentMoE、MTP、模型—硬件协同 | OpenMDW 发布可含权重、数据与配方；大型号依赖多卡和 NVIDIA 优化栈，需评估可移植性 |

补充说明：Grok 可用于分析“同一家族不同代际许可证完全不同”，但不作为本章架构主线；ERNIE 与 InternLM 分别适合异构多模态 MoE、科学领域化扩展阅读。

### 2.2．候选集的约束筛选

1. 明确输入/输出模态、语言、结构化输出、工具调用和延迟 SLO。
2. 用许可证、地域、数据驻留、衍生训练和再分发规则淘汰不合规候选。
3. 按总权重、激活权重、精度、上下文和并发计算资源，不用参数名中的 `A3B/A22B` 代替显存估算。
4. 在完成约束筛选后，于同一业务评测集、同一模板、同一采样设置和同一服务后端上比较质量、TTFT、TPOT、吞吐与成本。


<!-- theory-math-contract:v1 -->
### 2.3．核心机制的语言与数学表达

Mixture-of-Experts（MoE）路由器为每个 Token 选择得分最高的 $k$ 个专家，并对其输出加权：

$$
p(e\mid x)=\operatorname{softmax}(W_rx),\qquad
\mathcal T_k(x)=\operatorname{TopK}_e\,p(e\mid x),\qquad
y=\sum_{e\in\mathcal T_k(x)}\tilde p_eE_e(x)
$$

其中，$x\in\mathbb{R}^{D}$ 是 Token 隐状态，$W_r\in\mathbb{R}^{E\times D}$ 是路由权重，$E$ 是专家数，$\tilde p_e$ 是 Top-$k$ 内重新归一化的权重。`my_topk_router` 对应路由选择，模型 `config.json` 记录专家数量与激活数量，生产 Expert Parallel Runtime 负责跨设备派发。该公式只说明通用稀疏路由语义，不能据此推断某个模型家族的实际路由损失、共享专家或通信实现。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

| 理论对象 | 最小公式 | 原理实现函数 | 生产实现 |
|---|---|---|---|
| RMSNorm | $y=x/\sqrt{\operatorname{mean}(x^2)+\epsilon}\odot w$ | `my_rms_norm` | `torch.nn.functional.rms_norm` 或模型专用 fused kernel |
| Top-k 路由 | $g=\operatorname{softmax}(Wx),\ I=\operatorname{TopK}(g,k)$ | `my_top_k_route` | `torch.softmax`、`torch.topk`；大规模由 MoE Runtime 融合 |
| GQA KV 元素数 | $2BLn_{kv}d_h$ | `my_kv_cache_elements` | 从 `AutoConfig` 读取 `num_key_value_heads` 后交给服务容量模型 |
| MoE 激活比例 | $k/E$ | `my_active_expert_ratio` | 从 `num_experts`、`num_experts_per_tok` 读取并结合路由负载测量 |

公式、原理实现与生产接口的对应关系包括：符号所指张量、实现位置、库函数默认值、数值或梯度对齐方式，以及生产 Kernel 是否保持数学语义。


In [ ]:
import torch
import torch.nn.functional as F

DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")
# hidden=8 仅缩小原理夹具；生产宽度必须读取检查点 Config。
HIDDEN_SIZE = 8
# 4 个 Expert 中每 Token 选择 2 个用于呈现 50% 理论激活比例；已训练模型必须服从其路由 Config，并另测负载与丢 Token。
EXPERTS = 4
TOP_K = 2
# 1e-6 防止均方接近零时除零；加载检查点时不得偏离其 RMSNorm epsilon。
EPSILON = 1e-6
# 该 GQA 场景以 batch=2、sequence=128、2 个 KV Head、head dim=4、2 层估算缓存；各维均线性影响容量。
BATCH = 2
SEQUENCE = 128
KV_HEADS = 2
HEAD_DIM = 4
LAYERS = 2
# 每元素 2 Byte 对应 BF16/FP16；量化 KV 必须改用实际存储宽度及元数据。
KV_BYTES_PER_ELEMENT = 2


def my_rms_norm(x: torch.Tensor, weight: torch.Tensor, eps: float) -> torch.Tensor:
    """沿最后一维执行均方根归一化，并应用可学习缩放权重。"""
    variance = x.pow(2).mean(dim=-1, keepdim=True)
    return x * torch.rsqrt(variance + eps) * weight


def my_top_k_route(hidden_states: torch.Tensor, router_weight: torch.Tensor, top_k: int):
    """计算每个 Token 的专家路由概率，返回 Top-k 专家索引及重新归一化权重。"""
    logits = hidden_states @ router_weight.T
    probabilities = torch.softmax(logits, dim=-1)
    weights, expert_indices = torch.topk(probabilities, k=top_k, dim=-1)
    return expert_indices, weights / weights.sum(dim=-1, keepdim=True)


def my_kv_cache_elements(
    batch: int,
    sequence: int,
    key_value_heads: int,
    head_dim: int,
    layers: int,
) -> int:
    """计算所有层 Key/Value Cache 的元素总数，不包含元素字节宽度。"""
    return 2 * batch * sequence * key_value_heads * head_dim * layers


def my_active_expert_ratio(num_experts: int, experts_per_token: int) -> float:
    """返回每个 Token 激活专家数占专家总数的比例。"""
    return experts_per_token / num_experts


# seed=42 只固定测试夹具，不用于模型选择；质量结论需采用预注册的多 Seed 结果。
torch.manual_seed(42)
# 输入 2×3×8 覆盖 batch、sequence 与 hidden 三条轴，同时控制实验规模。
# 固定形状：x.shape = [2, 3, 8]。
x = torch.randn(2, 3, HIDDEN_SIZE, device=DEVICE)
weight = torch.ones(HIDDEN_SIZE, device=DEVICE)
# 固定形状：router_weight.shape = [4, 8]。
router_weight = torch.randn(EXPERTS, HIDDEN_SIZE, device=DEVICE)

manual = my_rms_norm(x, weight, EPSILON)
library = F.rms_norm(x, (HIDDEN_SIZE,), weight, EPSILON)
# FP32 的 rtol=1e-5、atol=1e-6 只接受舍入误差；更换低精度或 Kernel 后应按参考误差分布重设。
torch.testing.assert_close(manual, library, rtol=1e-5, atol=1e-6)
expert_indices, route_weights = my_top_k_route(x, router_weight, TOP_K)
kv_cache_elements = my_kv_cache_elements(BATCH, SEQUENCE, KV_HEADS, HEAD_DIM, LAYERS)

{
    "rms_norm_max_abs_error": float((manual - library).abs().max()),
    "route_shape": tuple(expert_indices.shape),
    "route_weight_sums": route_weights.sum(dim=-1),
    "active_expert_ratio": my_active_expert_ratio(EXPERTS, TOP_K),
    "kv_cache_elements": kv_cache_elements,
    "kv_cache_bytes": kv_cache_elements * KV_BYTES_PER_ELEMENT,
}


### 3.1．Top-k Expert 路由与负载分布

学习问题是：每个 Token 如何只激活部分 Expert，以及相同 Top-k 比例下各 Expert 的实际负载是否均衡。下图直接展开上一单元的 `expert_indices` 与 `route_weights`。验收条件是每个 Token 恰有 `TOP_K` 个非零路由项，保留项权重和为 1。


In [ ]:
# 将真实 Top-k 结果还原为 Token × Expert 稀疏矩阵，并统计 Expert 负载。
import matplotlib.pyplot as plt

flat_indices = expert_indices.detach().cpu().reshape(-1, TOP_K)
flat_weights = route_weights.detach().float().cpu().reshape(-1, TOP_K)
routing_matrix = torch.zeros(flat_indices.size(0), EXPERTS, dtype=torch.float32)
routing_matrix.scatter_(1, flat_indices, flat_weights)
nonzero_routes = routing_matrix.gt(0).sum(dim=1)
row_sum_error = float((routing_matrix.sum(dim=1) - 1.0).abs().max())
if not bool(nonzero_routes.eq(TOP_K).all()) or row_sum_error > 1e-6:
    raise RuntimeError("Top-k 路由矩阵不满足激活数或归一化契约")
expert_loads = routing_matrix.gt(0).sum(dim=0)
token_labels = [f"b{batch_index}:t{token_index}" for batch_index in range(x.size(0)) for token_index in range(x.size(1))]

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
image = axes[0].imshow(routing_matrix, cmap="viridis", vmin=0.0, vmax=1.0, aspect="auto")
for token_index in range(routing_matrix.size(0)):
    for expert_index in range(EXPERTS):
        if routing_matrix[token_index, expert_index] > 0:
            axes[0].text(expert_index, token_index, f"{routing_matrix[token_index, expert_index]:.2f}", ha="center", va="center", color="white")
axes[0].set(
    title="Token → Expert 稀疏路由权重", xlabel="Expert", ylabel="Token",
    xticks=range(EXPERTS), xticklabels=[f"E{i}" for i in range(EXPERTS)],
    yticks=range(len(token_labels)), yticklabels=token_labels,
)
bars = axes[1].bar(range(EXPERTS), expert_loads, color="#0072B2")
axes[1].bar_label(bars)
axes[1].set(title="每个 Expert 接收的 Token 数", xlabel="Expert", ylabel="Top-k 分配次数", xticks=range(EXPERTS), xticklabels=[f"E{i}" for i in range(EXPERTS)])
fig.colorbar(image, ax=axes[0], shrink=0.8, label="归一化路由权重")
plt.tight_layout()
plt.show()
print({"routing_shape": tuple(routing_matrix.shape), "max_row_sum_error": row_sum_error, "expert_loads": expert_loads.tolist()})


`TOP_K / EXPERTS` 只给出每个 Token 的理论激活比例，右图说明它不保证 Expert 间负载均衡。当前 Router 为随机初始化且只有 6 个 Token，因此不能解释为 Expert 专业化证据；生产还需在真实流量上统计容量溢出、丢 Token、共享 Expert、负载均衡 Loss、Expert Parallel 通信与尾延迟。


## 4．证据验证

运行结果应同时满足：

1. `my_rms_norm` 与 `torch.nn.functional.rms_norm` 的最大绝对误差接近当前精度的舍入范围。
2. Top-k 路由输出形状为 `[batch, sequence, TOP_K]`，最后一维权重和为 1。
3. 将 `num_key_value_heads` 从 MHA 的头数改为 GQA 的 KV 头数后，缓存元素数按比例下降。
4. `TOP_K / EXPERTS` 只表示理论激活比例；生产仍需测每个专家的 Token 分布、溢出和跨卡通信。

任一证据未满足均表明张量语义尚未对齐，此时不具备开展模型家族与后端比较的基础。


## 5．迁移到生产库

### 5.1．Qwen 与 DeepSeek：代表性结构案例

- **Qwen**：使用 Qwen3 小模型读取 Decoder-only、RoPE、GQA、Dense/MoE 等配置；Qwen3.5/3.6 的 Gated DeltaNet 与全注意力混合、原生视觉能力仅纳入架构图谱，不在本章重复实现。
- **DeepSeek**：用 V2-Lite 的 Config 观察 MLA/DeepSeekMoE，用 V3 理解无辅助损失负载均衡与 MTP，用 R1 理解推理后训练；当前 V4 只作为带日期的演进观察。
- **重要辨析**：`DeepSeek-R1-Distill-Qwen-*` 是把推理行为蒸馏到 Qwen 等网络架构的检查点，不能用于证明 DeepSeek MLA 或 DeepSeekMoE。

需要视觉输入时应使用 `AutoProcessor` 和相应的多模态 AutoModel。Chat Template、Thinking 开关、Tool Call 格式、EOS 协议和 Processor 均属于模型制品，不得依据统一 Prompt 推断。


### 5.2．Kimi、GLM 与 MiniMax：品牌谱系与架构差异

- **Kimi** 从 K2 到 K3 都以大规模稀疏 MoE 和 Agent 能力为重要方向，但注意力、残差、量化和许可证已经变化；必须逐检查点读取模型卡与 LICENSE。
- **GLM** 的历史谱系与现代 GLM-5 的 MoE/稀疏注意力不能混写；旧代训练目标不是新代永久不变的架构定义。
- **MiniMax** 适合观察线性/稀疏注意力与原生多模态演进，亦可用于分析自定义社区许可和 `trust_remote_code` 的供应链审计。

推理轨迹 SFT、RLVR、验证器和测试时计算统一回链 `A30_reasoning_model.ipynb`；专家并行与推理优化分别回链 `A40_training_optimization.ipynb` 与 `A50_inference_optimization.ipynb`。


### 5.3．Llama、Gemma、Phi、gpt-oss 与 Nemotron

- **Llama** 是自定义社区许可的开放权重家族；Llama 4 的原生图文与 MoE 不能被旧版 Llama 的部署经验完全替代。
- **Gemma** 覆盖端侧到服务器、Dense 到 MoE 和多模态路线；许可证也随代际变化，必须绑定具体模型卡。
- **Phi** 适合资源受限、端侧或专项任务，但“小”不代表可跳过语言、知识截止、安全和远程代码评测。
- **gpt-oss** 是 Apache-2.0 的文本开放权重模型；Harmony 格式、Reasoning Effort 和 Tool Schema 是正确行为的一部分。
- **Nemotron** 适合观察 Mamba/Attention/MoE 混合和硬件协同；完整开放程度与运行依赖仍按具体发布物核验。


### 5.4．模型仓库文件与证据层级

下载目录不是单一权重文件，而是一组共同定义模型身份、结构、输入协议和运行边界的版本化资产。审计结论必须区分发布者声明、机器可读配置、实际张量和运行时代码；其中任何一层都不能单独证明模型可以进入评估或部署。

| 证据层 | 代表文件 | 可确认的信息 | 证据边界 |
|---|---|---|---|
| 身份与治理 | `README.md`、`LICENSE*`、仓库文件哈希 | 来源、用途、基座、数据与评测声明、许可条款 | 属于发布者声明；缺失字段不能从权重反推 |
| 架构配置 | `config.json`、`generation_config.json` | 模型类型、层数、维度、Attention/MoE、上下文与默认生成策略 | 配置可能过期或依赖自定义代码，必须与张量核对 |
| 输入协议 | `tokenizer.json`、`tokenizer.model`、`tokenizer_config.json`、`chat_template.jinja`、Processor 配置 | 词表、特殊 Token、消息模板和多模态预处理 | 模板正确不代表模型质量或工具权限安全 |
| 权重事实 | `*.safetensors`、`*.safetensors.index.json` | 张量名称、形状、数据类型、分片归属和存储规模 | 量化打包元素数不等于原始参数量 |
| 派生与执行 | `adapter_config.json`、量化配置、`modeling_*.py`、`configuration_*.py` | 基座绑定、量化契约、自定义 Forward 与加载逻辑 | 远程代码需要独立审查；不得因模型可加载而自动信任 |

本节以 `OPEN_MODEL_DIR` 指向完整的本地模型目录。Hugging Face Cache 的 Snapshot 通常包含指向 `blobs` 目录的符号链接；本审计不会跟随这些链接，因此应先将当前所需文件物化为独立目录，再把该目录设为审计输入。`OPEN_MODEL_SOURCE` 记录来源，逐文件 SHA-256 记录实际内容；未提供来源字段时，审计结果保持阻断状态。文件摘要采用流式读取，不将权重载入内存。`OPEN_MODEL_HASH_WEIGHTS=0` 只适合快速排查，不能形成发布证据。


In [ ]:
from hashlib import sha256
from pathlib import Path
import json
import os
from tqdm.auto import tqdm

OPEN_MODEL_DIR = Path(
    os.environ.get("OPEN_MODEL_DIR", "/content/open_model")
).expanduser().resolve()
OPEN_MODEL_SOURCE = os.environ.get("OPEN_MODEL_SOURCE")
HASH_WEIGHT_FILES = os.environ.get("OPEN_MODEL_HASH_WEIGHTS", "1") == "1"
# 单个 JSON、文本元数据与 Safetensors Header 以 64 MiB 为审计资源上限；合法超限制品需隔离复核后显式调整并记录。
MAX_METADATA_BYTES = 64 * 1024 * 1024
MAX_SAFETENSORS_HEADER_BYTES = 64 * 1024 * 1024
# 8 MiB 分块仅影响哈希 I/O；关闭权重哈希只能用于排查，Manifest 仍保持阻断。
HASH_CHUNK_BYTES = 8 * 1024 * 1024

WEIGHT_SUFFIXES = {".safetensors", ".bin", ".pt", ".pth", ".ckpt", ".gguf", ".onnx"}
PICKLE_SUFFIXES = {".bin", ".pt", ".pth", ".ckpt", ".pickle", ".pkl"}


def my_reject_duplicate_pairs(pairs):
    """将 JSON 键值对构造成字典，并在发现重复键时抛出 ValueError。"""
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate JSON key: {key}")
        result[key] = value
    return result


def my_is_within(path: Path, root: Path) -> bool:
    """判断路径是否位于给定根目录内，不访问或修改文件。"""
    try:
        path.relative_to(root)
        return True
    except ValueError:
        return False


def my_regular_file(path: Path) -> Path:
    """解析并验证模型目录内的普通文件；符号链接、越界路径或非文件会触发异常。"""
    if path.is_symlink():
        raise ValueError(f"symlink is not read: {path}")
    resolved = path.resolve(strict=True)
    if not my_is_within(resolved, OPEN_MODEL_DIR) or not resolved.is_file():
        raise ValueError(f"path is outside the model directory or not a file: {path}")
    return resolved


def my_read_bytes_bounded(path: Path, limit: int = MAX_METADATA_BYTES) -> bytes:
    """在文件大小不超过上限时读取全部字节，否则抛出 ValueError。"""
    resolved = my_regular_file(path)
    size = resolved.stat().st_size
    if size > limit:
        raise ValueError(f"metadata exceeds {limit} bytes: {path.name} ({size})")
    return resolved.read_bytes()


def my_read_text_bounded(path: Path, limit: int = MAX_METADATA_BYTES) -> str:
    """按严格 UTF-8 解码有界文件内容，非法编码会触发异常。"""
    return my_read_bytes_bounded(path, limit).decode("utf-8", errors="strict")


def my_read_json_bounded(path: Path, limit: int = MAX_METADATA_BYTES):
    """读取有大小上限的 JSON，并拒绝包含重复键的对象。"""
    return json.loads(
        my_read_text_bounded(path, limit),
        object_pairs_hook=my_reject_duplicate_pairs,
    )


def my_sha256(path: Path) -> str:
    """分块读取模型目录内的普通文件并返回 SHA-256 十六进制摘要。"""
    resolved = my_regular_file(path)
    digest = sha256()
    with resolved.open("rb") as file, tqdm(
        total=resolved.stat().st_size, desc=f"SHA-256 {path.name}", unit="B",
        unit_scale=True, unit_divisor=1024, leave=False, dynamic_ncols=True,
    ) as progress:
        for chunk in iter(lambda: file.read(HASH_CHUNK_BYTES), b""):
            digest.update(chunk)
            progress.update(len(chunk))
    return digest.hexdigest()


def my_file_role(path: Path) -> str:
    """根据文件名与扩展名判定治理、权重、Tokenizer、配置、自定义代码或其他角色。"""
    name = path.name.lower()
    governance_name = name.replace("-", "_")
    if governance_name.startswith(("readme", "license", "copying", "notice", "aup", "acceptable_use", "use_policy", "usage_policy")):
        return "governance"
    if path.suffix.lower() in WEIGHT_SUFFIXES or name.endswith(".safetensors.index.json"):
        return "weights"
    if "tokenizer" in name or name in {"vocab.json", "merges.txt", "added_tokens.json", "special_tokens_map.json", "chat_template.jinja"}:
        return "tokenizer_or_template"
    if name == "config.json" or name.endswith("_config.json"):
        return "configuration"
    if path.suffix.lower() == ".py":
        return "custom_code"
    return "other"


def my_symlink_record(path: Path, model_dir: Path) -> dict[str, object]:
    """生成符号链接的仓库清单记录，并标记其目标是否越出模型目录。"""
    resolved_target = path.resolve(strict=False)
    return {
        "path": path.relative_to(model_dir).as_posix(),
        "kind": "symlink",
        "role": my_file_role(path),
        "target": os.readlink(path),
        "external_target": not my_is_within(resolved_target, model_dir),
        "size_bytes": None,
        "sha256": None,
        "pickle_serialization": path.suffix.lower() in PICKLE_SUFFIXES,
    }


def my_repository_inventory(model_dir: Path) -> list[dict[str, object]]:
    """遍历模型目录并生成文件、目录链接的角色、大小、哈希和序列化风险清单。"""
    records = []
    for current, directory_names, file_names in os.walk(
        model_dir, topdown=True, followlinks=False
    ):
        current_path = Path(current)
        retained_directories = []
        for directory_name in sorted(directory_names):
            path = current_path / directory_name
            if directory_name in {".git", ".cache"}:
                continue
            if path.is_symlink():
                records.append(my_symlink_record(path, model_dir))
            else:
                retained_directories.append(directory_name)
        directory_names[:] = retained_directories
        for file_name in sorted(file_names):
            path = current_path / file_name
            if path.is_symlink():
                records.append(my_symlink_record(path, model_dir))
                continue
            if not path.is_file():
                continue
            role = my_file_role(path)
            should_hash = HASH_WEIGHT_FILES or role != "weights"
            records.append(
                {
                    "path": path.relative_to(model_dir).as_posix(),
                    "kind": "file",
                    "role": role,
                    "size_bytes": path.stat().st_size,
                    "sha256": my_sha256(path) if should_hash else None,
                    "external_target": False,
                    "pickle_serialization": path.suffix.lower() in PICKLE_SUFFIXES,
                }
            )
    return sorted(records, key=lambda record: record["path"])


repository_inventory = (
    my_repository_inventory(OPEN_MODEL_DIR) if OPEN_MODEL_DIR.is_dir() else []
)
external_symlinks = [
    record["path"] for record in repository_inventory if record.get("external_target")
]
repository_summary = {
    "status": (
        "blocked_external_symlink"
        if external_symlinks
        else "observed" if repository_inventory else "blocked_missing_directory"
    ),
    "model_dir": str(OPEN_MODEL_DIR),
    "entry_count": len(repository_inventory),
    "total_regular_file_bytes": sum(
        record.get("size_bytes") or 0 for record in repository_inventory
    ),
    "symlinks_followed": False,
    "external_symlinks": external_symlinks,
    "weight_hashes_complete": HASH_WEIGHT_FILES,
}
repository_summary


### 5.5．从 `config.json` 还原模型结构

`model_type` 决定 Transformers 的配置家族，`architectures` 表示保存时声明的任务模型类；二者不能替代 `modeling_*.py` 中的实际计算过程。结构审计至少读取层数、隐藏维度、Attention Head、KV Head、MLP、上下文、RoPE、滑窗、MoE、权重共享、数据类型、量化配置和 `auto_map`。多模态模型还需分别审计 `text_config`、`vision_config`、`audio_config` 与连接器配置。

若 `AutoConfig` 在 `trust_remote_code=False` 时无法识别模型，结果表示当前库版本缺少安全的内置映射，而不是授权执行仓库代码。此时应计算文件摘要并静态审阅配置类和模型类，再在隔离环境中决定是否允许自定义代码。


In [ ]:
from importlib.metadata import PackageNotFoundError, version
from transformers import AutoConfig

CONFIG_FIELDS = (
    "model_type", "architectures", "vocab_size", "hidden_size",
    "num_hidden_layers", "num_attention_heads", "num_key_value_heads",
    "head_dim", "intermediate_size", "max_position_embeddings",
    "rope_theta", "rope_scaling", "sliding_window",
    "tie_word_embeddings", "torch_dtype", "quantization_config", "auto_map",
)


def my_first_config_value(config_data: dict[str, object], names: tuple[str, ...]):
    """按候选字段顺序返回配置中第一个已声明的值，均不存在时返回 None。"""
    for name in names:
        if name in config_data:
            return config_data[name]
    return None


def my_config_summary(config_data: dict[str, object], identity: dict[str, object]):
    """从文本配置和仓库身份信息提取结构参数，并派生头维度、GQA 与 MoE 摘要。"""
    text_config = config_data.get("text_config")
    structural = text_config if isinstance(text_config, dict) else config_data
    summary = {field: structural.get(field, config_data.get(field)) for field in CONFIG_FIELDS}
    hidden_size = my_first_config_value(structural, ("hidden_size", "d_model", "n_embd"))
    attention_heads = my_first_config_value(structural, ("num_attention_heads", "n_head"))
    key_value_heads = structural.get("num_key_value_heads")
    head_dim = (
        hidden_size // attention_heads
        if isinstance(hidden_size, int) and isinstance(attention_heads, int)
        and attention_heads > 0 and hidden_size % attention_heads == 0
        else None
    )
    query_heads_per_kv = (
        attention_heads // key_value_heads
        if isinstance(attention_heads, int) and isinstance(key_value_heads, int)
        and key_value_heads > 0 and attention_heads % key_value_heads == 0
        else None
    )
    summary.update(
        {
            "identity": identity,
            "hidden_size": hidden_size,
            "num_hidden_layers": my_first_config_value(structural, ("num_hidden_layers", "n_layer")),
            "num_attention_heads": attention_heads,
            "head_dim_derived": head_dim,
            "query_heads_per_kv_head_derived": query_heads_per_kv,
            "num_experts": my_first_config_value(
                structural, ("num_experts", "n_routed_experts", "num_local_experts")
            ),
            "experts_per_token": my_first_config_value(
                structural, ("num_experts_per_tok", "num_experts_per_token", "moe_top_k")
            ),
            "sub_configs": sorted(
                key for key in ("text_config", "vision_config", "audio_config")
                if isinstance(config_data.get(key), dict)
            ),
        }
    )
    return summary


config_path = OPEN_MODEL_DIR / "config.json"
raw_config = {}
config_read_error = None
if config_path.exists():
    try:
        parsed_config = my_read_json_bounded(config_path)
        if not isinstance(parsed_config, dict):
            raise ValueError("config.json root must be an object")
        raw_config = parsed_config
    except Exception as error:
        config_read_error = f"{type(error).__name__}: {error}"

try:
    transformers_version = version("transformers")
except PackageNotFoundError:
    transformers_version = None

if raw_config:
    try:
        local_config_object = AutoConfig.from_pretrained(
            OPEN_MODEL_DIR, local_files_only=True, trust_remote_code=False
        )
        library_view = {
            "status": "recognized_without_repository_code",
            "config_class": type(local_config_object).__name__,
            "transformers_version": transformers_version,
        }
    except Exception as error:
        library_view = {
            "status": "not_recognized_without_repository_code",
            "config_class": None,
            "transformers_version": transformers_version,
            "error": f"{type(error).__name__}: {error}",
        }
    local_config_audit = my_config_summary(
        raw_config,
        {
            "source": OPEN_MODEL_SOURCE,
            "static_json_sha256": my_sha256(config_path),
            "library_view": library_view,
        },
    )
else:
    local_config_audit = {
        "status": "blocked_invalid_or_missing_config",
        "error": config_read_error,
    }
local_config_audit


### 5.6．Tokenizer、特殊 Token 与 Chat Template

Tokenizer 决定文本如何进入网络，Chat Template 决定消息角色、工具描述和生成起始位置如何编码。审计应同时记录基础词表大小、加入特殊 Token 后的实际长度、BOS/EOS/PAD/UNK 映射、Added Token、模型输入字段和模板摘要。`config.json` 的 `vocab_size`、Tokenizer 实际长度与 Embedding 行数需要联合核对；不一致只有在明确记录新增 Token 和 Embedding 扩展时才可接受。

模板文件可以证明输入序列的构造规则，但不能证明模型已按该模板训练，也不能授予工具执行权限。多模态模型还应检查 `AutoProcessor` 所依赖的图像、音频、视频预处理配置。


In [ ]:
def my_optional_json(name: str):
    """尝试读取模型目录中的可选 JSON，返回数据与错误信息二元组。"""
    path = OPEN_MODEL_DIR / name
    if not path.exists():
        return None, None
    try:
        return my_read_json_bounded(path), None
    except Exception as error:
        return None, f"{name}: {type(error).__name__}: {error}"


def my_component_type(value):
    """从字典组件声明中提取 type 字段，其他输入返回 None。"""
    return value.get("type") if isinstance(value, dict) else None


def my_template_evidence(source: str, value) -> dict[str, object]:
    """把模板值规范化后生成来源、类型、字符数与 SHA-256 证据。"""
    canonical = (
        value
        if isinstance(value, str)
        else json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    )
    return {
        "source": source,
        "value_type": type(value).__name__,
        "character_count": len(canonical),
        "sha256": sha256(canonical.encode("utf-8")).hexdigest(),
        "rendered": False,
    }


def my_tokenizer_summary(model_dir: Path) -> dict[str, object]:
    """审计本地 Tokenizer 与 Chat Template 资产，汇总配置、特殊 Token、模板来源及解析错误。"""
    tokenizer_files = sorted(
        record["path"] for record in repository_inventory
        if record["role"] == "tokenizer_or_template" and record["kind"] == "file"
    )
    if not tokenizer_files:
        return {"status": "blocked_missing_tokenizer_assets", "files": []}

    errors = []
    tokenizer_config, error = my_optional_json("tokenizer_config.json")
    if error:
        errors.append(error)
    special_tokens_map, error = my_optional_json("special_tokens_map.json")
    if error:
        errors.append(error)
    tokenizer_json, error = my_optional_json("tokenizer.json")
    if error:
        errors.append(error)
    vocab_json, error = my_optional_json("vocab.json")
    if error:
        errors.append(error)
    added_tokens_json, error = my_optional_json("added_tokens.json")
    if error:
        errors.append(error)

    tokenizer_config = tokenizer_config if isinstance(tokenizer_config, dict) else {}
    special_tokens_map = special_tokens_map if isinstance(special_tokens_map, dict) else {}
    tokenizer_json = tokenizer_json if isinstance(tokenizer_json, dict) else {}
    vocab_json = vocab_json if isinstance(vocab_json, (dict, list)) else None
    model_section = tokenizer_json.get("model")
    model_section = model_section if isinstance(model_section, dict) else {}
    vocab = model_section.get("vocab")
    base_vocab_source = "tokenizer.json:model.vocab"
    if not isinstance(vocab, (dict, list)) and isinstance(vocab_json, (dict, list)):
        vocab = vocab_json
        base_vocab_source = "vocab.json"
    base_vocab_size = len(vocab) if isinstance(vocab, (dict, list)) else None
    token_ids = []
    if isinstance(vocab, dict):
        token_ids.extend(value for value in vocab.values() if type(value) is int)
    elif isinstance(vocab, list):
        token_ids.extend(range(len(vocab)))
    added_tokens = tokenizer_json.get("added_tokens")
    added_token_ids = []
    if isinstance(added_tokens, list):
        added_token_ids.extend(
            item.get("id") for item in added_tokens
            if isinstance(item, dict) and type(item.get("id")) is int
        )
    if isinstance(added_tokens_json, dict):
        added_token_ids.extend(
            value for value in added_tokens_json.values() if type(value) is int
        )
    elif isinstance(added_tokens_json, list):
        added_token_ids.extend(
            item.get("id") for item in added_tokens_json
            if isinstance(item, dict) and type(item.get("id")) is int
        )
    added_tokens_decoder = tokenizer_config.get("added_tokens_decoder")
    if isinstance(added_tokens_decoder, dict):
        for token_id in added_tokens_decoder:
            if isinstance(token_id, str) and token_id.isdecimal():
                added_token_ids.append(int(token_id))
            elif type(token_id) is int:
                added_token_ids.append(token_id)
    added_token_ids = sorted(set(value for value in added_token_ids if value >= 0))
    token_ids.extend(added_token_ids)
    unique_token_ids = sorted(set(token_ids))
    vocab_id_contiguous = (
        unique_token_ids == list(range(unique_token_ids[-1] + 1))
        if unique_token_ids else None
    )
    effective_vocab_size = (
        unique_token_ids[-1] + 1
        if vocab_id_contiguous
        else base_vocab_size if vocab_id_contiguous is None else None
    )

    special_tokens = dict(special_tokens_map)
    for name in ("bos_token", "eos_token", "pad_token", "unk_token"):
        if name in tokenizer_config and name not in special_tokens:
            special_tokens[name] = tokenizer_config[name]

    limitations = []
    if (model_dir / "tokenizer.model").exists():
        limitations.append(
            "tokenizer.model 的二进制词表与归一化规则未被静态还原；其与 JSON 资产的一致性不可验证"
        )
    if base_vocab_size is None:
        limitations.append("未发现可静态解析的 tokenizer.json:model.vocab 或 vocab.json")

    templates = []
    if "chat_template" in tokenizer_config:
        templates.append(
            my_template_evidence("tokenizer_config.json:chat_template", tokenizer_config["chat_template"])
        )
    template_path = model_dir / "chat_template.jinja"
    if template_path.exists():
        try:
            templates.append(
                my_template_evidence("chat_template.jinja", my_read_text_bounded(template_path))
            )
        except Exception as error:
            errors.append(f"chat_template.jinja: {type(error).__name__}: {error}")

    return {
        "status": (
            "failed_static_parse" if errors
            else "observed_static" if base_vocab_size is not None
            else "not_verifiable"
        ),
        "parsing_mode": "static_files_only",
        "tokenizer_class_declared": tokenizer_config.get("tokenizer_class"),
        "algorithm": model_section.get("type"),
        "base_vocab_size": base_vocab_size,
        "base_vocab_source": base_vocab_source if base_vocab_size is not None else None,
        "effective_vocab_size": effective_vocab_size,
        "vocab_id_contiguous": vocab_id_contiguous,
        "added_token_count": len(added_token_ids),
        "added_token_ids": added_token_ids,
        "normalizer": my_component_type(tokenizer_json.get("normalizer")),
        "pre_tokenizer": my_component_type(tokenizer_json.get("pre_tokenizer")),
        "post_processor": my_component_type(tokenizer_json.get("post_processor")),
        "decoder": my_component_type(tokenizer_json.get("decoder")),
        "special_tokens": special_tokens,
        "special_token_ids_declared_by_model_config": {
            name: raw_config.get(f"{name}_id")
            for name in ("bos_token", "eos_token", "pad_token", "unk_token")
        },
        "chat_templates": templates,
        "chat_template_rendered": False,
        "files": tokenizer_files,
        "limitations": limitations,
        "errors": errors,
    }


tokenizer_audit = (
    my_tokenizer_summary(OPEN_MODEL_DIR)
    if OPEN_MODEL_DIR.is_dir()
    else {"status": "blocked_missing_directory", "files": []}
)
tokenizer_audit


### 5.7．Safetensors、权重索引与张量结构

Safetensors 文件头包含张量名称、形状、数据类型和数据区偏移，可以在不创建 Tensor、不占用模型内存的条件下读取。分片模型还应解析 `model.safetensors.index.json` 一类索引，验证每个张量只映射到一个分片、所有分片均存在、索引记录与实际文件头一致，并确认数据区不存在重叠或未索引空洞。

张量名称形成实际模块树。例如 `model.layers.0.self_attn.q_proj.weight` 同时提供层编号、Attention 投影位置和矩阵形状。层数、Embedding、LM Head、Q/K/V、MLP 和 Expert 张量应与 Config 相互验证。对于量化权重，打包 Tensor、Scale 与 Zero Point 都会进入文件头，因此存储元素总数不能直接解释为原始参数量。


In [ ]:
from collections import Counter
from math import prod
import re

_LAYER_PATTERN = re.compile(r"(?:^|\.)(?:layers|h|blocks|block)\.(\d+)(?:\.|$)")
_DTYPE_BITS = {
    "BOOL": 8, "U8": 8, "I8": 8, "F8_E4M3": 8, "F8_E5M2": 8,
    "F8_E8M0": 8, "U16": 16, "I16": 16, "F16": 16, "BF16": 16,
    "U32": 32, "I32": 32, "F32": 32, "U64": 64, "I64": 64,
    "F64": 64, "U4": 4, "I4": 4, "F4": 4,
}


def my_safetensors_header(path: Path) -> dict[str, object]:
    """有界解析 Safetensors 文件头，验证元数据范围并返回张量描述。"""
    resolved = my_regular_file(path)
    file_size = resolved.stat().st_size
    with resolved.open("rb") as file:
        prefix = file.read(8)
        if len(prefix) != 8:
            raise ValueError(f"{path.name}: missing 8-byte header length")
        header_size = int.from_bytes(prefix, byteorder="little", signed=False)
        if header_size <= 0 or header_size > MAX_SAFETENSORS_HEADER_BYTES:
            raise ValueError(f"{path.name}: invalid header size {header_size}")
        if header_size > file_size - 8:
            raise ValueError(f"{path.name}: header exceeds file size")
        header_bytes = file.read(header_size)
        if len(header_bytes) != header_size:
            raise ValueError(f"{path.name}: truncated header")
        if not header_bytes.startswith(b"{"):
            raise ValueError(f"{path.name}: header must begin with a JSON object")
    header = json.loads(
        header_bytes.decode("utf-8", errors="strict"),
        object_pairs_hook=my_reject_duplicate_pairs,
    )
    if not isinstance(header, dict):
        raise ValueError(f"{path.name}: header root must be an object")

    payload_size = file_size - 8 - header_size
    tensors = {}
    spans = []
    for name, metadata in header.items():
        if name == "__metadata__":
            if not isinstance(metadata, dict) or not all(
                isinstance(key, str) and isinstance(value, str)
                for key, value in metadata.items()
            ):
                raise ValueError(f"{path.name}: __metadata__ must map strings to strings")
            continue
        if not isinstance(name, str) or not isinstance(metadata, dict):
            raise ValueError(f"{path.name}: invalid tensor entry")
        dtype = metadata.get("dtype")
        shape = metadata.get("shape")
        offsets = metadata.get("data_offsets")
        if dtype not in _DTYPE_BITS:
            raise ValueError(f"{path.name}:{name}: unsupported dtype {dtype!r}")
        if not isinstance(shape, list) or not all(type(value) is int and value >= 0 for value in shape):
            raise ValueError(f"{path.name}:{name}: invalid shape")
        if (
            not isinstance(offsets, list) or len(offsets) != 2
            or not all(type(value) is int for value in offsets)
            or not 0 <= offsets[0] <= offsets[1] <= payload_size
        ):
            raise ValueError(f"{path.name}:{name}: invalid data offsets")
        numel = prod(shape)
        expected_bytes = (numel * _DTYPE_BITS[dtype] + 7) // 8
        stored_bytes = offsets[1] - offsets[0]
        if stored_bytes != expected_bytes:
            raise ValueError(
                f"{path.name}:{name}: byte range {stored_bytes} != expected {expected_bytes}"
            )
        spans.append((offsets[0], offsets[1], name))
        tensors[name] = {
            "shape": shape, "dtype": dtype, "data_offsets": offsets,
            "numel": numel, "stored_bytes": stored_bytes,
            "shard": path.relative_to(OPEN_MODEL_DIR).as_posix(),
        }
    previous_end = 0
    previous_name = None
    for start, end, name in sorted(spans):
        if end == start:
            continue
        if start < previous_end:
            raise ValueError(f"{path.name}: overlapping tensors {previous_name!r} and {name!r}")
        if start > previous_end:
            raise ValueError(f"{path.name}: unindexed payload gap before {name!r}")
        previous_end, previous_name = end, name
    if previous_end != payload_size:
        raise ValueError(f"{path.name}: payload is not entirely indexed")
    return {
        "file": path.relative_to(OPEN_MODEL_DIR).as_posix(),
        "file_size": file_size,
        "header_size": header_size,
        "header_sha256": sha256(header_bytes).hexdigest(),
        "payload_size": payload_size,
        "metadata": header.get("__metadata__", {}),
        "tensors": tensors,
    }


def my_weight_summary(model_dir: Path) -> dict[str, object]:
    """汇总全部 Safetensors 分片的张量、形状、数据类型、层索引和一致性问题。"""
    shard_records = [
        record for record in repository_inventory
        if record["kind"] == "file" and record["path"].endswith(".safetensors")
    ]
    if not shard_records:
        return {
            "status": "not_verifiable",
            "reason": "no readable local Safetensors shard; use a format-specific parser",
            "tensors": {}, "issues": [],
        }

    tensor_records = {}
    tensors_by_shard = {}
    issues = []
    shard_headers = []
    for record in shard_records:
        shard_path = model_dir / record["path"]
        try:
            parsed = my_safetensors_header(shard_path)
        except Exception as error:
            issues.append(f"{record['path']}: {type(error).__name__}: {error}")
            continue
        tensors_by_shard[parsed["file"]] = set(parsed["tensors"])
        shard_headers.append({key: value for key, value in parsed.items() if key != "tensors"})
        for name, metadata in parsed["tensors"].items():
            if name in tensor_records:
                issues.append(f"duplicate tensor across shards: {name}")
                continue
            tensor_records[name] = metadata

    index_records = [
        record for record in repository_inventory
        if record["kind"] == "file" and record["path"].endswith(".safetensors.index.json")
    ]
    if len(shard_records) > 1 and not index_records:
        issues.append("multiple Safetensors shards exist without an index")
    index_summaries = []
    for record in index_records:
        try:
            index_data = my_read_json_bounded(model_dir / record["path"])
            weight_map = index_data.get("weight_map") if isinstance(index_data, dict) else None
            if not isinstance(weight_map, dict) or not all(
                isinstance(name, str) and isinstance(shard, str) for name, shard in weight_map.items()
            ):
                raise ValueError("weight_map must be a string-to-string object")
            referenced_shards = set(weight_map.values())
            missing_shards = sorted(referenced_shards - set(tensors_by_shard))
            missing_from_headers = sorted(
                name for name, shard in weight_map.items()
                if shard not in tensors_by_shard or name not in tensors_by_shard[shard]
            )
            unindexed = sorted(
                name for shard in referenced_shards & set(tensors_by_shard)
                for name in tensors_by_shard[shard] if weight_map.get(name) != shard
            )
            if missing_shards:
                issues.append(f"{record['path']}: missing or unreadable shards {missing_shards}")
            if missing_from_headers:
                issues.append(f"{record['path']}: {len(missing_from_headers)} index tensors absent from mapped headers")
            if unindexed:
                issues.append(f"{record['path']}: {len(unindexed)} header tensors absent from reverse index coverage")
            index_summaries.append(
                {"file": record["path"], "tensor_count": len(weight_map),
                 "referenced_shards": sorted(referenced_shards),
                 "missing_shards": missing_shards,
                 "missing_from_headers": missing_from_headers, "unindexed": unindexed}
            )
        except Exception as error:
            issues.append(f"{record['path']}: {type(error).__name__}: {error}")

    layer_indices = sorted(
        {int(match.group(1)) for name in tensor_records if (match := _LAYER_PATTERN.search(name))}
    )
    catalog = [
        [name, item["shape"], item["dtype"], item["shard"]]
        for name, item in sorted(tensor_records.items())
    ]
    return {
        "status": "observed" if tensor_records and not issues else "failed_consistency",
        "shards": shard_headers, "indexes": index_summaries,
        "tensor_count": len(tensor_records),
        "stored_elements": sum(item["numel"] for item in tensor_records.values()),
        "stored_bytes": sum(item["stored_bytes"] for item in tensor_records.values()),
        "dtype_counts": dict(Counter(item["dtype"] for item in tensor_records.values())),
        "layer_indices": layer_indices,
        "tensor_catalog_sha256": sha256(
            json.dumps(catalog, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
        ).hexdigest(),
        "issues": issues, "tensors": tensor_records,
    }


weight_audit = (
    my_weight_summary(OPEN_MODEL_DIR)
    if OPEN_MODEL_DIR.is_dir()
    else {"status": "blocked_missing_directory", "tensors": {}, "issues": []}
)
{key: value for key, value in weight_audit.items() if key != "tensors"}


### 5.8．Adapter、量化格式、多模态资产与自定义代码

| 变体或文件 | 必须确认的关系 | 生产处置 |
|---|---|---|
| LoRA/Adapter | `base_model_name_or_path`、基座模型 ID、文件哈希、Target Module、秩与组合顺序 | Adapter 不能作为完整模型独立解释；基座与 Adapter 分别保留摘要 |
| GPTQ/AWQ/BitsAndBytes/FP8 | 位宽、Group Size、对称性、校准方法、目标 Kernel 与量化工具版本 | `quantization_config` 只声明格式；必须在目标后端验证可消费性和质量回退 |
| GGUF/ONNX/MLX 等派生格式 | 原始模型 ID、文件哈希、转换工具、转换参数、容器内元数据 | 使用格式专用解析器；不得套用 Safetensors 参数统计方法 |
| 多模态 Processor | 图像尺寸、归一化、帧率、音频采样率、模态特殊 Token | Processor 与权重作为同一制品版本发布 |
| `auto_map` 与 Python 文件 | 配置类、模型类、Tokenizer/Processor 与动态导入入口 | 固定代码摘要，完成静态审查和隔离验证后才允许执行 |
| `.bin/.pt/.pth/.ckpt` | 是否依赖 Pickle 反序列化、是否存在等价 Safetensors | 未知来源文件不得直接反序列化；优先取得或转换为安全格式 |

文件存在只能证明制品包含某项声明，不能证明许可证有效、量化无质量损失或自定义代码安全。相应结论需要法务、评测、后端兼容和安全证据共同完成。


In [ ]:
import ast

adapter_config, adapter_config_error = my_optional_json("adapter_config.json")
separate_quantization_config, quantization_config_error = my_optional_json(
    "quantization_config.json"
)
adapter_config = adapter_config if isinstance(adapter_config, dict) else None
separate_quantization_config = (
    separate_quantization_config if isinstance(separate_quantization_config, dict) else None
)
quantization_config = (
    raw_config.get("quantization_config") if raw_config else None
) or separate_quantization_config
custom_python_files = sorted(
    record["path"]
    for record in repository_inventory
    if record["role"] == "custom_code" and record["kind"] == "file"
)
unsafe_serialization_files = sorted(
    record["path"]
    for record in repository_inventory
    if record["pickle_serialization"]
)
processor_files = sorted(
    record["path"]
    for record in repository_inventory
    if any(
        marker in record["path"].lower()
        for marker in ("processor", "preprocessor", "image_processor", "feature_extractor")
    )
)
license_files = sorted(
    record["path"]
    for record in repository_inventory
    if record["role"] == "governance"
    and record["kind"] == "file"
    and Path(record["path"]).name.lower().startswith(("license", "copying"))
)
usage_policy_files = sorted(
    record["path"]
    for record in repository_inventory
    if record["role"] == "governance" and record["kind"] == "file"
    and Path(record["path"]).name.lower().replace("-", "_").startswith(
        ("aup", "acceptable_use", "use_policy", "usage_policy")
    )
)
weight_formats = sorted(
    {
        Path(record["path"]).suffix.lower()
        for record in repository_inventory
        if record["role"] == "weights" and record["kind"] == "file"
    }
)


def my_static_python_summary(relative_path: str) -> dict[str, object]:
    """仅通过 AST 静态解析自定义 Python 文件，返回哈希、类名和解析状态而不执行代码。"""
    path = OPEN_MODEL_DIR / relative_path
    try:
        source = my_read_text_bounded(path)
        tree = ast.parse(source, filename=relative_path)
        return {
            "path": relative_path,
            "sha256": my_sha256(path),
            "classes": sorted(
                node.name for node in ast.walk(tree) if isinstance(node, ast.ClassDef)
            ),
            "status": "parsed_without_execution",
        }
    except Exception as error:
        return {
            "path": relative_path, "classes": [], "status": "failed_static_parse",
            "error": f"{type(error).__name__}: {error}",
        }


python_code_audit = {
    path: my_static_python_summary(path) for path in custom_python_files
}


def my_auto_map_checks(auto_map) -> list[dict[str, object]]:
    """检查 auto_map 声明的模块与类是否存在于本地静态解析结果中。"""
    if not isinstance(auto_map, dict):
        return []
    checks = []
    for auto_class, declared_targets in sorted(auto_map.items()):
        targets = declared_targets if isinstance(declared_targets, list) else [declared_targets]
        for declared_target in targets:
            if declared_target is None:
                continue
            if not isinstance(declared_target, str):
                checks.append({"auto_class": auto_class, "target": declared_target, "status": "fail", "reason": "target is not a string"})
                continue
            local_target = declared_target.rsplit("--", 1)[-1]
            if "." not in local_target:
                checks.append({"auto_class": auto_class, "target": declared_target, "status": "fail", "reason": "target has no module/class separator"})
                continue
            module_name, class_name = local_target.rsplit(".", 1)
            module_parts = module_name.split(".")
            if not class_name.isidentifier() or not all(part.isidentifier() for part in module_parts):
                checks.append({"auto_class": auto_class, "target": declared_target, "status": "fail", "reason": "invalid module or class identifier"})
                continue
            relative_module = Path(*module_parts).with_suffix(".py").as_posix()
            code_summary = python_code_audit.get(relative_module)
            status = (
                "pass" if code_summary and code_summary["status"] == "parsed_without_execution"
                and class_name in code_summary["classes"] else "fail"
            )
            checks.append(
                {"auto_class": auto_class, "target": declared_target,
                 "module_file": relative_module, "class_name": class_name,
                 "status": status, "repository_code_executed": False}
            )
    return checks


auto_map = raw_config.get("auto_map") if raw_config else None
auto_map_audit = my_auto_map_checks(auto_map)
model_card_files = sorted(
    record["path"] for record in repository_inventory
    if record["kind"] == "file" and Path(record["path"]).name.lower().startswith("readme")
)

variant_audit = {
    "adapter": adapter_config,
    "adapter_base_model": (
        adapter_config.get("base_model_name_or_path") if adapter_config else None
    ),
    "quantization": quantization_config,
    "processor_files": processor_files,
    "adapter_config_error": adapter_config_error,
    "quantization_config_error": quantization_config_error,
    "custom_python_files": custom_python_files,
    "python_code_static_audit": python_code_audit,
    "auto_map": auto_map,
    "auto_map_static_checks": auto_map_audit,
    "repository_code_executed": False,
    "unsafe_serialization_files": unsafe_serialization_files,
    "license_files": license_files,
    "usage_policy_files": usage_policy_files,
    "model_card_files": model_card_files,
    "weight_formats": weight_formats,
}
variant_audit


### 5.9．配置—Tokenizer—权重一致性验证

一致性检查采用 `pass`、`fail` 与 `not_verifiable` 三态。只有实际观察到的证据才能通过；命名规则无法识别或证据缺失时保留“不可验证”，不能将其解释为一致。通用检查包括 Config 层数与层索引、词表大小与 Embedding、隐藏维度与 Embedding 列数、Tokenizer 有效长度与 Embedding 行数、权重索引双向覆盖、外部符号链接、`auto_map` 静态目标以及 Adapter 基座绑定。

张量命名不是跨模型家族的统一标准，因此自动检查只负责发现明确冲突。MLA、混合 Attention、共享 Expert、跨模态连接器和量化打包等结构仍需结合对应模型类进行人工核验。


In [ ]:
def my_tensor_by_suffix(
    tensors: dict[str, dict[str, object]], suffixes: tuple[str, ...]
):
    """按后缀优先级查找权重张量，返回名称与元数据；未找到时返回两个 None。"""
    for suffix in suffixes:
        for name in sorted(tensors):
            if name.endswith(suffix):
                return name, tensors[name]
    return None, None


def my_check(name: str, status: str, evidence: dict[str, object]):
    """构造统一的一致性检查记录，并拒绝不受支持的状态值。"""
    if status not in {"pass", "fail", "not_verifiable"}:
        raise ValueError(f"unsupported check status: {status}")
    return {"name": name, "status": status, "evidence": evidence}


def my_consistency_checks() -> list[dict[str, object]]:
    """交叉核对 Config、Tokenizer 与权重元数据，返回可验证、失败或不可验证的证据列表。"""
    checks = []
    tensors = weight_audit.get("tensors", {})
    structural = raw_config.get("text_config") if raw_config else None
    if not isinstance(structural, dict):
        structural = raw_config

    declared_layers = my_first_config_value(
        structural, ("num_hidden_layers", "n_layer")
    ) if structural else None
    observed_layers = weight_audit.get("layer_indices", [])
    if isinstance(declared_layers, int) and observed_layers:
        checks.append(
            my_check(
                "config_vs_weight_layers",
                "pass" if observed_layers == list(range(declared_layers)) else "fail",
                {"declared": declared_layers, "observed_indices": observed_layers},
            )
        )
    else:
        checks.append(my_check("config_vs_weight_layers", "not_verifiable", {}))

    embedding_name, embedding = my_tensor_by_suffix(
        tensors,
        ("embed_tokens.weight", "tok_embeddings.weight", "wte.weight", "word_embeddings.weight"),
    )
    vocab_size = structural.get("vocab_size") if structural else None
    hidden_size = my_first_config_value(
        structural, ("hidden_size", "d_model", "n_embd")
    ) if structural else None
    if embedding and len(embedding["shape"]) == 2:
        rows, columns = embedding["shape"]
        checks.append(
            my_check(
                "config_vs_embedding_vocab",
                "pass" if vocab_size == rows else "fail",
                {"tensor": embedding_name, "config": vocab_size, "rows": rows},
            )
            if isinstance(vocab_size, int)
            else my_check("config_vs_embedding_vocab", "not_verifiable", {})
        )
        checks.append(
            my_check(
                "config_vs_embedding_hidden",
                "pass" if hidden_size == columns else "fail",
                {"tensor": embedding_name, "config": hidden_size, "columns": columns},
            )
            if isinstance(hidden_size, int)
            else my_check("config_vs_embedding_hidden", "not_verifiable", {})
        )
        tokenizer_length = tokenizer_audit.get("effective_vocab_size")
        checks.append(
            my_check(
                "tokenizer_vs_embedding",
                "pass" if tokenizer_length == rows else "fail",
                {"tokenizer_length": tokenizer_length, "embedding_rows": rows},
            )
            if isinstance(tokenizer_length, int)
            else my_check("tokenizer_vs_embedding", "not_verifiable", {})
        )
    else:
        checks.extend(
            [
                my_check("config_vs_embedding_vocab", "not_verifiable", {}),
                my_check("config_vs_embedding_hidden", "not_verifiable", {}),
                my_check("tokenizer_vs_embedding", "not_verifiable", {}),
            ]
        )

    weight_status = weight_audit.get("status")
    checks.append(
        my_check(
            "safetensors_headers_and_index",
            "pass" if weight_status == "observed"
            else "not_verifiable" if weight_status == "not_verifiable" else "fail",
            {"status": weight_status, "issues": weight_audit.get("issues", [])},
        )
    )
    checks.append(
        my_check(
            "external_symlink_boundary",
            "fail" if external_symlinks else "pass",
            {"external_symlinks": external_symlinks, "symlinks_followed": False},
        )
    )
    checks.append(
        my_check(
            "auto_map_static_targets",
            "pass" if not auto_map or (
                auto_map_audit and all(item["status"] == "pass" for item in auto_map_audit)
            ) else "fail",
            {"declared": bool(auto_map), "checks": auto_map_audit, "code_executed": False},
        )
    )
    if adapter_config:
        adapter_base = adapter_config.get("base_model_name_or_path")
        checks.append(
            my_check(
                "adapter_base_identity",
                "pass" if adapter_base else "fail",
                {"base_model": adapter_base},
            )
        )
    else:
        checks.append(
            my_check("adapter_base_identity", "pass", {"not_applicable": True})
        )
    return checks


consistency_checks = my_consistency_checks()
{
    "checks": consistency_checks,
    "failures": [check for check in consistency_checks if check["status"] == "fail"],
    "not_verifiable": [
        check for check in consistency_checks if check["status"] == "not_verifiable"
    ],
}


### 5.10．`Model Audit Manifest`

`Model Audit Manifest` 是模型进入评估与部署前的不可变证据索引。它至少绑定来源、Commit SHA、文件路径与摘要、Config 摘要、Tokenizer/Template 摘要、权重文件头、分片一致性、Adapter 基座、量化与 Processor 配置、自定义代码清单、自动一致性检查和人工复核状态。规范化 JSON 的摘要用于识别 Manifest 自身，原始文件和完整报告仍需保留在受控制品库中。

自动审计不能确认训练数据完整性、训练配方真实性、许可证法律效力、评测污染、模型能力或恶意行为。缺失信息必须记录为未知或阻断条件。Manifest 使用 `blocked`、`conditional` 与 `eligible_for_runtime_validation` 三种状态；最后一种只表示静态证据允许进入受控运行时验证，不等于获得生产发布批准。

第三方开放权重模型进入 `50_model_evaluation.ipynb` 时绑定该 Manifest；`60_inference_deployment.ipynb` 将通过评估的精确制品置于受控部署环境；`70_model_safety.ipynb` 完成供应链、访问控制、运行时防线与事件响应验收后，才形成最终生产放量结论。更新模型前应取得新的 Commit SHA，并重新审阅文件、配置、模板、代码和权重索引差异。


In [ ]:
from datetime import datetime, timezone

weight_file_records = [
    record for record in repository_inventory
    if record["kind"] == "file" and record["role"] == "weights"
]
internal_symlinks = [
    record["path"] for record in repository_inventory
    if record["kind"] == "symlink" and not record["external_target"]
]
static_code_failures = [
    path for path, result in python_code_audit.items()
    if result["status"] != "parsed_without_execution"
]
hard_blockers = []
conditional_requirements = []
if not repository_inventory:
    hard_blockers.append("model_directory_not_observed")
if not OPEN_MODEL_SOURCE:
    hard_blockers.append("source_missing")
if external_symlinks:
    hard_blockers.append("external_symlink_blocked")
if not raw_config:
    hard_blockers.append("config_missing_or_invalid")
if tokenizer_audit.get("status") != "observed_static":
    hard_blockers.append("tokenizer_static_audit_incomplete")
if weight_audit.get("status") != "observed":
    hard_blockers.append("weight_header_or_index_audit_incomplete")
if not weight_file_records or any(record.get("sha256") is None for record in weight_file_records):
    hard_blockers.append("weight_hash_coverage_incomplete")
if unsafe_serialization_files:
    hard_blockers.append("pickle_based_weights_require_isolated_conversion")
if not license_files:
    hard_blockers.append("license_evidence_missing")
if not model_card_files:
    hard_blockers.append("model_card_missing")
if static_code_failures:
    hard_blockers.append("custom_code_static_parse_failed")
if any(check["status"] == "fail" for check in consistency_checks):
    hard_blockers.append("static_consistency_failed")
if adapter_config and (
    not variant_audit.get("adapter_base_model")
):
    hard_blockers.append("adapter_base_identity_incomplete")
if internal_symlinks:
    conditional_requirements.append("internal_symlinks_not_followed")
if custom_python_files:
    conditional_requirements.append("custom_code_requires_isolated_review")
if quantization_config:
    conditional_requirements.append("quantization_requires_quality_and_kernel_validation")
if processor_files:
    conditional_requirements.append("processor_requires_multimodal_contract_validation")
if adapter_config:
    conditional_requirements.append("adapter_requires_base_and_composition_validation")
if any(check["status"] == "not_verifiable" for check in consistency_checks):
    conditional_requirements.append("static_evidence_contains_not_verifiable_items")

audit_status = (
    "blocked" if hard_blockers
    else "conditional" if conditional_requirements
    else "eligible_for_runtime_validation"
)
weight_summary = {
    key: value for key, value in weight_audit.items() if key != "tensors"
}
model_audit_manifest = {
    "schema_version": "1.0",
    "manifest_type": "open_model_audit",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "identity": {
        "source": OPEN_MODEL_SOURCE,
        "local_directory": str(OPEN_MODEL_DIR),
    },
    "source_evidence": {
        "model_card_files": model_card_files,
        "license_files": license_files,
        "usage_policy_files": usage_policy_files,
    },
    "files": repository_inventory,
    "config": local_config_audit,
    "tokenizer": tokenizer_audit,
    "weights": weight_summary,
    "variants": variant_audit,
    "consistency_checks": consistency_checks,
    "audit_method": {
        "network_access": False,
        "tensor_payload_loaded": False,
        "repository_code_executed": False,
        "pickle_deserialized": False,
        "tokenizer_instantiated": False,
        "chat_template_rendered": False,
        "symlinks_followed": False,
        "external_symlinks_blocked": bool(external_symlinks),
        "metadata_limit_bytes": MAX_METADATA_BYTES,
        "safetensors_header_limit_bytes": MAX_SAFETENSORS_HEADER_BYTES,
        "hash_chunk_bytes": HASH_CHUNK_BYTES,
        "weight_hash_coverage": (
            "complete"
            if weight_file_records
            and all(record.get("sha256") for record in weight_file_records)
            else "incomplete"
        ),
    },
    "gate": {
        "status": audit_status,
        "blockers": sorted(set(hard_blockers)),
        "conditions": sorted(set(conditional_requirements)),
        "downstream_reviews": [
            "license_and_aup_legal_review",
            "model_and_system_evaluation",
            "runtime_compatibility_and_capacity",
            "security_acceptance_and_release_authorization",
        ],
    },
}
canonical_manifest = json.dumps(
    model_audit_manifest,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")
model_audit_manifest["manifest_sha256"] = sha256(canonical_manifest).hexdigest()


def my_export_model_audit_manifest(manifest: dict[str, object], output_path: str | Path):
    """以独占创建方式将审计 Manifest 写到模型目录之外，并返回路径与内容哈希。"""
    output = Path(output_path).expanduser().resolve()
    if my_is_within(output, OPEN_MODEL_DIR):
        raise ValueError("store the audit manifest outside the audited model directory")
    if not output.parent.is_dir():
        raise FileNotFoundError(f"output directory does not exist: {output.parent}")
    payload = json.dumps(
        manifest, ensure_ascii=False, sort_keys=True, indent=2
    ) + "\n"
    with output.open("x", encoding="utf-8", newline="\n") as file:
        file.write(payload)
    return {
        "path": str(output),
        "sha256": sha256(payload.encode("utf-8")).hexdigest(),
    }


{
    "status": audit_status,
    "blockers": model_audit_manifest["gate"]["blockers"],
    "conditions": model_audit_manifest["gate"]["conditions"],
    "manifest_sha256": model_audit_manifest["manifest_sha256"],
    "audit_method": model_audit_manifest["audit_method"],
}


## 6．生产边界

静态仓库审计只建立模型制品的身份和结构证据，不替代质量评估、受控部署或安全验收。生产路线固定为：

```mermaid
flowchart LR
    A["E10：开放权重仓库审计"] --> E["50：模型与系统评估"]
    E --> D["60：受控部署与运行时验证"]
    D --> S["70：安全验收与最终生产放量"]
    S --> R["持续监控、事件响应与回滚"]
```

| 阶段 | 消费的证据 | 阶段结论 | 不能越过的边界 |
|---|---|---|---|
| E10 仓库审计 | Repo ID、文件 SHA-256、模型卡、许可证、Config、Tokenizer、权重头、代码摘要 | `blocked`、`conditional` 或 `eligible_for_runtime_validation` | 不输出模型质量、业务价值或生产发布结论 |
| 50 模型评估 | Model Audit Manifest、冻结数据集、切片、指标与生成配置 | 可审计的离线准入证据 | 未通过离线硬门禁不能进入部署 |
| 60 受控部署 | 通过评估的精确制品、目标 Runtime、硬件与容量模型 | 隔离或灰度环境中的兼容性、性能与回滚证据 | 受控部署不等于对外生产放量；最低身份、密钥、权限、网络隔离和回滚门禁仍是部署前置 |
| 70 模型安全 | E10 供应链证据、50 风险评测、60 运行时拓扑与 Trace | 完整威胁建模、安全控制、事件响应与最终放量结论 | 安全控制应回写评测和部署门禁，不能解释为上线后补做安全 |

### 6.1．制品与解析边界

- 未知来源的 `.bin`、`.pt`、`.pth` 和 `.ckpt` 可能依赖 Pickle 反序列化；静态审计只记录文件，不调用 `torch.load` 或等价接口。
- Safetensors 降低了反序列化执行风险，但仍需固定解析器版本、限制 Header 大小、验证偏移与字节区间，并在隔离环境完成后续加载。
- `auto_map` 和本地 Python 文件属于可执行供应链。AST 只能核验语法、模块路径和类声明，不能证明代码无恶意行为；执行前仍需代码审查、依赖锁定、无凭据构建与网络隔离。
- Tokenizer 与 Chat Template 的静态摘要不能证明训练协议相容。受控加载后还需比较特殊 Token、固定样例 Token ID、单步 Logits 和生成终止行为。
- GGUF、ONNX、MLX 等派生格式需要格式专用解析器，并绑定原始模型 ID、文件哈希、转换工具、转换参数和目标 Runtime；不能套用 Safetensors 的参数统计结论。
- 模型卡、许可证、AUP、后端兼容矩阵和 Kernel 支持均具有版本与时间边界。任何文件、依赖或运行时变化都会使既有 Manifest 失效并触发重新审计。

### 6.2．生产验收顺序

生产验收依次固定来源并计算文件哈希，核验模型卡、许可证与 AUP，生成仓库清单和全量摘要，解析 Config、Tokenizer、Chat Template、Safetensors 与 Index，审查 Adapter、量化、Processor 和自定义代码，再执行配置—代码—权重一致性检查。静态门禁通过后，先进行受控加载、单步 Logits 与固定样例验证；随后进入 `50_model_evaluation.ipynb` 完成离线评估，再由 `60_inference_deployment.ipynb` 完成后端兼容、服务压测以及隔离影子流量或受限灰度，最后由 `70_model_safety.ipynb` 完成安全验收与最终生产放量。

官方接口参考：[Transformers Auto Classes](https://huggingface.co/docs/transformers/model_doc/auto)、[Safetensors](https://huggingface.co/docs/safetensors/)、[vLLM Supported Models](https://docs.vllm.ai/en/latest/models/supported_models/) 与 [SGLang 文档](https://docs.sglang.ai/)。
